#### Técnicas de Otimização

Quando você trabalha com grande volumes de dados é possível armazenar essas informações para reutilização de forma rapida e segura assim otimizado as suas transformações e limpeza com esses dados.

Existem duas funções que são essenciais para desempenha esse papel

    cache() = Armazena o fluxo na memoria ram, quando o volume de dados é entre média a pequeno é possível alocar na memoria ram...

    persist() = Armazena o fluxo na memoria ram em conjunto com disco da maquima, podendo salvar grande volume de dados

Assim evitando realizar recomputação dos dados em uma nova variavel otimizado e melhorando o desempenho.

In [1]:
# Importando Bibliotecas para uso do Spark para usar dentro do VSCODE
import findspark
findspark.init()
from pyspark.sql import SparkSession # Import para inciar o spark
from pyspark.sql.types import * # Todos as configurações do SQL de tipagem no spark
from pyspark.sql.functions import * # Todas as funções de SQL do Spark
# Criação da sessão Spark
spark = SparkSession.builder.appName("PySpark-VSCode-App").getOrCreate()
print("PySpark está pronto para uso!")

PySpark está pronto para uso!


Vamos realizar a leitura de arquivo em csv com poucos registros para apresenta a melhoria na execução ele estando em cache e não estando...

In [12]:
arquivo_csv = r'C:\Users\Cleydenilson\Documents\Scripts\01 - APRENDIZADOS\07_PySpark\Data\V_OCORRENCIA_AMPLA.csv'
df_csv = (
    spark.read
    .option("header", "true") # definido que o primeiro registro deve ser o cabeçalho
    .option("sep", ";") # Definido o delimitador padrão do arquivo
    .option("comment", "A")   # ignora linhas iniciadas com "A"
    .option("multiLine", "true") # Realizado leitura ignorando dados com quebras de linhas
    .option("escape", '"')     # Tratamento dos textos dentro do arquivo csv
    .option("quote", '"')      # Ignorando informações com aspas duplas
    .csv(arquivo_csv)
)


In [ ]:
# Realizado a contagem de regisros que temos no dataframe
df_csv.count()

5021

Como pode ver levou em torno de 0.8 segundos para me retorna a quantidade de registros que tem nesse dataframe.

In [ ]:
# Fazendo que que nosso dataframe seja salvo em cache na memoria ram
df_csv.cache()

DataFrame[Numero_da_Ocorrencia: string, Numero_da_Ficha: string, Operador_Padronizado: string, Classificacao_da_Ocorrencia: string, Data_da_Ocorrencia: string, Hora_da_Ocorrencia: string, Municipio: string, UF: string, Regiao: string, Descricao_do_Tipo: string, ICAO: string, Latitude: string, Longitude: string, Tipo_de_Aerodromo: string, Historico: string, Matricula: string, Categoria_da_Aeronave: string, Operador: string, Tipo_de_Ocorrencia: string, Fase_da_Operacao: string, Operacao: string, Danos_a_Aeronave: string, Aerodromo_de_Destino: string, Aerodromo_de_Origem: string, Lesoes_Fatais_Tripulantes: string, Lesoes_Fatais_Passageiros: string, Lesoes_Fatais_Terceiros: string, Lesoes_Graves_Tripulantes: string, Lesoes_Graves_Passageiros: string, Lesoes_Graves_Terceiros: string, Lesoes_Leves_Tripulantes: string, Lesoes_Leves_Passageiros: string, Lesoes_Leves_Terceiros: string, Ilesos_Tripulantes: string, Ilesos_Passageiros: string, Lesoes_Desconhecidas_Tripulantes: string, Lesoes_Desco

In [ ]:
# Realizado novamente a contagem só para avaliar a evolução na perfomance após o armazenamento em cache do arquivo.
df_csv.count()

5021

Ao armazena-lo em cache temos uma otimização para 0.4 ou seja **50%** mais rapido o processamento.

#### **Persit()**

No persit conseguimos manipular a forma que desejamos realizar o armazenamento dos nossos dados, os mais comuns são:

    MEMORY_ONLY: Mantém apenas na memória ram igual o cache.
    MEMORY_AND_DISK: Mantém de forma hibrida uma parte em memoria ram e outra no disco.
    DISK_ONLY: Tudo no disco (Armazena volume de dados muito grandes, porém tem uma perfomance mais lenta)
    MEMORY_ONLY_SER: Serializa em memória (Mais eficiente com memoria ram)
    OFF_HEAP: Uso da memória fora da JVM (Menos Comum, porém as vezes usual)

In [21]:
from pyspark import StorageLevel # Import necessário para poder usar os nível de armazenamento explicando acima

df_csv.persist(StorageLevel.MEMORY_AND_DISK)

DataFrame[Numero_da_Ocorrencia: string, Numero_da_Ficha: string, Operador_Padronizado: string, Classificacao_da_Ocorrencia: string, Data_da_Ocorrencia: string, Hora_da_Ocorrencia: string, Municipio: string, UF: string, Regiao: string, Descricao_do_Tipo: string, ICAO: string, Latitude: string, Longitude: string, Tipo_de_Aerodromo: string, Historico: string, Matricula: string, Categoria_da_Aeronave: string, Operador: string, Tipo_de_Ocorrencia: string, Fase_da_Operacao: string, Operacao: string, Danos_a_Aeronave: string, Aerodromo_de_Destino: string, Aerodromo_de_Origem: string, Lesoes_Fatais_Tripulantes: string, Lesoes_Fatais_Passageiros: string, Lesoes_Fatais_Terceiros: string, Lesoes_Graves_Tripulantes: string, Lesoes_Graves_Passageiros: string, Lesoes_Graves_Terceiros: string, Lesoes_Leves_Tripulantes: string, Lesoes_Leves_Passageiros: string, Lesoes_Leves_Terceiros: string, Ilesos_Tripulantes: string, Ilesos_Passageiros: string, Lesoes_Desconhecidas_Tripulantes: string, Lesoes_Desco

In [22]:
df_csv.count()

5021

Analise que ele executou exatamente em 0.0 segundos um ganho de perfomance de 100% mais performatico

### Funções repartition() e coalesce()

São funções que vão mudar o número de partições de um Dataframe/RDD, Isso afeta diretamente:

    O Paralelismo (Quantos nós ou núcleos, trabalham ao mesmo tempo)

    O uso de rede e disco

    A performance de leitura, trasnformação e escrita.

Principais diferenças entre as funções


    Função: repartition()
    Aumenta nº de Partições?: Sim
    Reduz o nº de Partições?: Sim
    Causa Shuffle completo?: Sim (Custo Alto)
    Quando usar?: Quando você quer mais partições ou uma redistribuição completa.

__________________________________

    Função: coalesce()
    Aumenta nº de Partições?: Não
    Reduz o nº de Partições?: Sim
    Causa Shuffle completo?: Não (Ou parcial)
    Quando usar?: Quando você quer reduzir partições com menos custos.



In [23]:
# Chamando repartition e dividindo o em 10 partições
df_csv.repartition(10)

DataFrame[Numero_da_Ocorrencia: string, Numero_da_Ficha: string, Operador_Padronizado: string, Classificacao_da_Ocorrencia: string, Data_da_Ocorrencia: string, Hora_da_Ocorrencia: string, Municipio: string, UF: string, Regiao: string, Descricao_do_Tipo: string, ICAO: string, Latitude: string, Longitude: string, Tipo_de_Aerodromo: string, Historico: string, Matricula: string, Categoria_da_Aeronave: string, Operador: string, Tipo_de_Ocorrencia: string, Fase_da_Operacao: string, Operacao: string, Danos_a_Aeronave: string, Aerodromo_de_Destino: string, Aerodromo_de_Origem: string, Lesoes_Fatais_Tripulantes: string, Lesoes_Fatais_Passageiros: string, Lesoes_Fatais_Terceiros: string, Lesoes_Graves_Tripulantes: string, Lesoes_Graves_Passageiros: string, Lesoes_Graves_Terceiros: string, Lesoes_Leves_Tripulantes: string, Lesoes_Leves_Passageiros: string, Lesoes_Leves_Terceiros: string, Ilesos_Tripulantes: string, Ilesos_Passageiros: string, Lesoes_Desconhecidas_Tripulantes: string, Lesoes_Desco

In [ ]:
# Realizado o count para avaliar a performance da execução
df_csv.count()

5021

Como pode ver a execução aumentou para 0.2 segundos, como falei causa o shuffle completo, causando um custo alto

In [25]:
# Realizado o coalesce para reduzir 2 particões 
df_csv = df_csv.coalesce(2)

In [26]:
# Realizado o count para avaliar a performance da execução
df_csv.count()

5021

Ao reduzir o número de partições em 2 temos um pequeno aumento na execução reduzindo para 0.1 segundos como pode avaliarem.